# Ray Tune only — CIFAR-10 ResNet18 hyperparameter tuning

Notebook zawiera tylko część do **Ray Tune**:
- jedna komórka z datasetem, modelem i mixupem,
- funkcja `objective(config)`,
- `tune.Tuner(...)`,
- odczyt najlepszego wyniku.

Nie ma tutaj `TorchTrainer`, `ScalingConfig`, `RunConfig`, `ray.train`, baseline treningu ani finalnego treningu.


In [1]:
!pip install -U "ray[tune]"


In [3]:
import os
import logging

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


def build_cifar10_datasets(data_dir="./data"):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        ),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        ),
    ])

    trainset = torchvision.datasets.CIFAR10(
        root=data_dir,
        train=True,
        download=False,   # <- ważne
        transform=transform_train,
    )

    testset = torchvision.datasets.CIFAR10(
        root=data_dir,
        train=False,
        download=False,   # <- ważne
        transform=transform_test,
    )

    return trainset, testset



classes = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)


def make_resnet18_cifar(num_classes=10):
    model = resnet18(weights=None)

    model.conv1 = nn.Conv2d(
        3,
        64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False,
    )

    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0

    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a = y
    y_b = y[idx]

    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def mixup_correct_count(pred_logits, y_a, y_b, lam):
    pred = pred_logits.argmax(dim=1)

    return (
        lam * (pred == y_a).float()
        + (1 - lam) * (pred == y_b).float()
    ).sum().item()


Device: cuda


In [4]:
DATA_DIR = "/tmp/cifar10_data"

_ = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
)

_ = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
)

print("CIFAR-10 downloaded / verified")

100%|██████████| 170M/170M [00:05<00:00, 30.8MB/s]


CIFAR-10 downloaded / verified


In [5]:
import ray
from ray import tune

logging.getLogger("ray").setLevel(logging.ERROR)

if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=2,
    num_gpus=1,
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
)

print("Ray version:", ray.__version__)
print("Ray resources:", ray.cluster_resources())


Ray version: 2.55.1
Ray resources: {'node:__internal_head__': 1.0, 'CPU': 2.0, 'object_store_memory': 3970129920.0, 'node:172.28.0.12': 1.0, 'memory': 9263636480.0, 'accelerator_type:T4': 1.0, 'GPU': 1.0}


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [6]:
def objective(config):
    local_device = "cuda" if torch.cuda.is_available() else "cpu"

    local_trainset, local_testset = build_cifar10_datasets("/tmp/cifar10_data")

    trainloader = torch.utils.data.DataLoader(
        local_trainset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
    )

    testloader = torch.utils.data.DataLoader(
        local_testset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
    )

    model = make_resnet18_cifar().to(local_device)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["label_smoothing"],
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config["epochs"],
    )

    for epoch in range(config["epochs"]):
        model.train()

        running_loss = 0.0
        correct_soft = 0.0
        total = 0

        for images, labels in trainloader:
            images = images.to(local_device)
            labels = labels.to(local_device)

            optimizer.zero_grad(set_to_none=True)

            if config["mixup_alpha"] > 0:
                images, y_a, y_b, lam = mixup_data(
                    images,
                    labels,
                    alpha=config["mixup_alpha"],
                )

                outputs = model(images)

                loss = mixup_criterion(
                    criterion,
                    outputs,
                    y_a,
                    y_b,
                    lam,
                )

                correct_soft += mixup_correct_count(
                    outputs,
                    y_a,
                    y_b,
                    lam,
                )
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

                correct_soft += (
                    outputs.argmax(dim=1) == labels
                ).float().sum().item()

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            total += labels.size(0)

        scheduler.step()

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in testloader:
                images = images.to(local_device)
                labels = labels.to(local_device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()
                val_total += labels.size(0)

        tune.report(
            {
                "epoch": epoch + 1,
                "train_loss": running_loss / total,
                "train_acc": 100.0 * correct_soft / total,
                "val_loss": val_loss / val_total,
                "val_acc": 100.0 * val_correct / val_total,
            }
        )


In [7]:
search_space = {
    "epochs": 5,
    "batch_size": tune.choice([64, 128]),
    "lr": tune.loguniform(1e-5, 3e-3),
    "weight_decay": tune.loguniform(1e-6, 1e-2),
    "label_smoothing": tune.choice([0.0, 0.05, 0.1]),
    "mixup_alpha": tune.choice([0.0, 0.2, 0.4]),
}

trainable = tune.with_resources(
    objective,
    resources={
        "cpu": 1,
        "gpu": 1,
    },
)

tuner = tune.Tuner(
    trainable,
    param_space=search_space,
    tune_config=tune.TuneConfig(
        metric="val_acc",
        mode="max",
        num_samples=3,
    ),
)

results = tuner.fit()


== Status ==
Current time: 2026-05-31 18:22:18 (running for 00:00:00.28)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
Result logdir: /tmp/ray/session_2026-05-31_18-22-04_682331_7058/artifacts/2026-05-31_18-22-17/objective_2026-05-31_18-22-17/driver_artifacts
Number of trials: 3/3 (3 PENDING)
+-----------------------+----------+-------+--------------+-------------------+-------------+---------------+----------------+
| Trial name            | status   | loc   |   batch_size |   label_smoothing |          lr |   mixup_alpha |   weight_decay |
|-----------------------+----------+-------+--------------+-------------------+-------------+---------------+----------------|
| objective_a7ecc_00000 | PENDING  |       |           64 |              0    | 5.81762e-05 |           0.2 |    2.34537e-06 |
| objective_a7ecc_00001 | PENDING  |       |          128 |              0.05 | 3.68477e-05 |           0.4 |    1.62334e-06 |
| objec

In [8]:
best_result = results.get_best_result(
    metric="val_acc",
    mode="max",
)

print("Best val_acc:", best_result.metrics["val_acc"])
print("Best config:")
best_result.config


Best val_acc: 82.21
Best config:


{'epochs': 5,
 'batch_size': 64,
 'lr': 0.0002357541861547472,
 'weight_decay': 1.0692935590937963e-06,
 'label_smoothing': 0.1,
 'mixup_alpha': 0.4}

In [9]:
results_df = results.get_dataframe()
results_df[
    [
        "config/batch_size",
        "config/lr",
        "config/weight_decay",
        "config/label_smoothing",
        "config/mixup_alpha",
        "val_acc",
        "val_loss",
        "train_acc",
        "train_loss",
        "epoch",
    ]
].sort_values("val_acc", ascending=False)


,config/batch_size,config/lr,config/weight_decay,config/label_smoothing,config/mixup_alpha,val_acc,val_loss,train_acc,train_loss,epoch
2,64,0.000236,0.000001,0.10,0.4,82.21,0.944428,68.372649,1.310551,5
0,64,0.000058,0.000002,0.00,0.2,70.46,0.862281,62.717351,1.158306,5
1,128,0.000037,0.000002,0.05,0.4,59.81,1.286339,50.517647,1.567529,5
